In [1]:
# Setup
import sys
sys.path.append('..')

from src.multi_file_loader import MultiFileLoader
from src.context_preserver import ContextPreserver
from src.summarizer import Summarizer
from src.config import Config
from pathlib import Path

print("✓ Imports successful")

✓ Imports successful


## Step 1: Load Configuration

In [2]:
# Load config and validate API access
config = Config()

if Config.AZURE_OPENAI_ENDPOINT:
    print(f"✓ Azure OpenAI endpoint: {Config.AZURE_OPENAI_ENDPOINT}")
    print(f"✓ Chat deployment: {Config.AZURE_OPENAI_CHAT_DEPLOYMENT}")
elif Config.OPENAI_API_KEY:
    print("✓ Standard OpenAI API key loaded")
else:
    print("⚠ No API key found. Set AZURE_OPENAI_API_KEY + AZURE_OPENAI_ENDPOINT in .env")
    print("  Summarization requires API access")


✓ Azure OpenAI endpoint: https://initiative.openai.azure.com/
✓ Chat deployment: gpt-4.1


## Step 2: Prepare Elements

In [3]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Load and link documents
loader = MultiFileLoader()
preserver = ContextPreserver()

data_folder = Path("../data")

if data_folder.exists():
    # Load documents
    documents = loader.load_all_documents(str(data_folder))
    print(f"✓ Loaded {len(documents)} documents")
    
    # Link elements — normalize keys for doc types that use 'content' instead of 'text_chunks'
    for doc in documents:
        text_chunks = doc.get('text_chunks', [])
        if not text_chunks and 'content' in doc:
            text_chunks = [{
                'element_id': f"{doc.get('file_id', 'unknown')}_text_0",
                'content': doc['content'],
                'type': 'text'
            }]
        if not text_chunks and not doc.get('tables', []) and not doc.get('images', []):
            continue
        preserver.add_document_elements(
            file_id=doc.get('file_id', ''),
            filename=doc.get('filename', ''),
            text_chunks=text_chunks,
            tables=doc.get('tables', []),
            images=doc.get('images', [])
        )
    
    elements = preserver.all_elements
    print(f"✓ Prepared {len(elements)} elements for summarization")
    
    # Show element types
    type_counts = {}
    for elem in elements:
        elem_type = elem['type']
        type_counts[elem_type] = type_counts.get(elem_type, 0) + 1
    
    print("\nElement types:")
    for elem_type, count in sorted(type_counts.items(), key=lambda x: -x[1]):
        print(f"  {elem_type}: {count}")
else:
    print("⚠ No data folder found")
    elements = []

Skipping code zBatchRollupReportIdGenerator.sj: Zero-byte file
Skipping code zCalculatedMeterEditor.sj: Zero-byte file
Skipping code zCFCalculation.sj: Zero-byte file
Skipping code zCFMigration.sj: Zero-byte file
Skipping code zCloseGroupScheduleDate.sj: Zero-byte file
Skipping code zContactDistributionList.sj: Zero-byte file
Skipping code zContactRelationship.sj: Zero-byte file
Skipping code zContactTypeEditor.sj: Zero-byte file
Skipping code zDataCondenser.sj: Zero-byte file
Skipping code zDataExchange.sj: Zero-byte file
Skipping code zDOTHazardEditor.sj: Zero-byte file
Skipping code zDurationFix.sj: Zero-byte file
Skipping code zEditFlowComputerMeters.sj: Zero-byte file
Skipping code zEditFlowComputerProduct.sj: Zero-byte file
Skipping code zEditReason.sj: Zero-byte file
Skipping code zEffeciencyDefinitionEditor.sj: Zero-byte file
Skipping code zEnterProductAlias.sj: Zero-byte file
Skipping code zExportDataEditor.sj: Zero-byte file
Skipping code zExternalVolumeDefaultValues.sj: Zero

✓ Loaded 22531 documents
✓ Prepared 123310 elements for summarization

Element types:
  text: 109066
  table: 13027
  image: 831
  table_header: 386


## Step 3: Initialize Summarizer

In [4]:
# Create summarizer with Azure OpenAI (gpt-4.1 — best model for summarization)
if Config.AZURE_OPENAI_ENDPOINT:
    summarizer = Summarizer(
        openai_api_key=Config.AZURE_OPENAI_API_KEY,
        azure_endpoint=Config.AZURE_OPENAI_ENDPOINT,
        api_version=Config.AZURE_OPENAI_API_VERSION,
        azure_deployment=Config.AZURE_OPENAI_CHAT_DEPLOYMENT  # gpt-4.1
    )
    print("✓ Summarizer initialized (Azure OpenAI)")
    print(f"  Deployment: {Config.AZURE_OPENAI_CHAT_DEPLOYMENT}")
    print("  Model: gpt-4.1 (text + vision)")
elif Config.OPENAI_API_KEY:
    summarizer = Summarizer(
        openai_api_key=Config.OPENAI_API_KEY,
        text_model="gpt-4.1",
        vision_model="gpt-4.1"
    )
    print("✓ Summarizer initialized (Standard OpenAI)")
    print("  Model: gpt-4.1 (text + vision)")
else:
    print("⚠ Cannot initialize summarizer without API key")
    summarizer = None


✓ Summarizer initialized (Azure OpenAI)
  Deployment: gpt-4.1
  Model: gpt-4.1 (text + vision)


## Step 4: Demonstrate Text Summarization

In [5]:
# Find a text element and summarize it
if summarizer and elements:
    # Find first text element
    text_elem = None
    for elem in elements:
        if elem['type'] == 'text' and len(elem.get('content', '')) > 100:
            text_elem = elem
            break
    
    if text_elem:
        print("Text Summarization Example:\n")
        print("=" * 80)
        
        # Original text
        original = text_elem['content']
        print(f"\nOriginal text ({len(original)} chars):")
        print(f"{original[:300]}..." if len(original) > 300 else original)
        
        # Generate summaries
        print("\nGenerating summaries...")
        summaries = summarizer.summarize_text(original)
        
        print(f"\n✓ Summaries generated:\n")
        print(f"Short ({len(summaries['short'])} chars):")
        print(f"  {summaries['short']}")
        print(f"\nMedium ({len(summaries['medium'])} chars):")
        print(f"  {summaries['medium']}")
        print(f"\nFull ({len(summaries['full'])} chars):")
        print(f"  {summaries['full'][:200]}..." if len(summaries['full']) > 200 else f"  {summaries['full']}")
        
        print("\n" + "=" * 80)
    else:
        print("No suitable text elements found")

Text Summarization Example:


Original text (194861 chars):
/// <reference path="../Scripts/jquery.js" />
/// <reference path="../Scripts/MadCapUtilities.js" />
/// <reference path="../Scripts/MadCapGlobal.js" />
/// <reference path="../Scripts/MadCapDom.js" />
/// <reference path="Scripts/MadCapHelpSystem.js" />

/*!
* Copyright MadCap Software
* http://www...

Generating summaries...

✓ Summaries generated:

Short (53 chars):
  JavaScript module loader setup and utility functions.

Medium (468 chars):
  This code snippet includes reference paths for jQuery and several MadCap scripts, and contains the RequireJS 2.1.11 library, which is used for JavaScript module loading and dependency management. It defines utility functions for type checking, object property handling, and module normalization, and sets up the core logic for resolving, loading, and managing JavaScript modules in a browser environment. The code is part of MadCap Software's v17.2.8047.30191 release.

Full (194861 chars)

## Step 5: Demonstrate Image Description

In [6]:
# Find an image and describe it
if summarizer and elements:
    # Find first image element
    image_elem = None
    for elem in elements:
        if elem['type'] == 'image':
            image_elem = elem
            break
    
    if image_elem:
        print("Image Description Example:\n")
        print("=" * 80)
        
        print(f"\nImage from: {image_elem.get('filename', 'Unknown')}")
        print(f"Page: {image_elem.get('page_number', 'N/A')}")
        
        # Describe image (content holds base64-encoded data)
        print("\nGenerating AI description...")
        description = summarizer.describe_image(
            image_base64=image_elem.get('content', ''),
            context=f"This image is from {image_elem.get('filename')}"
        )
        
        print(f"\n✓ Description generated:\n")
        print(f"{description}")
        
        print("\n" + "=" * 80)
    else:
        print("No image elements found")

Image Description Example:


Image from: 01.01.03.01 - Importing gas and liquid meter text files by service - TC.docx
Page: None

Generating AI description...

✓ Description generated:

{'short': 'Gas and liquid meter import configuration panel.', 'full': 'Certainly! Here is a detailed description of the image:\n\n---\n\n### 1. What the image shows\nThe image is a screenshot of a software interface used for configuring the import of gas and liquid meter text files by service. It appears to be part of a utility or data import application, likely used in an industrial or utility context.\n\n---\n\n### 2. Key visual elements\n- **Tabs at the top:**  \n  There are three tabs visible:\n  - **File Imports** (selected)\n  - TESTit Imports\n  - Liquid Imports\n\n- **Input fields and labels:**  \n  - **Folder to Check:**  \n    A text box with the path `D:\\FILEIMPORTS-10-5-0` is entered. Next to it is a greyed-out "Browse..." button.\n  - **After every run pause for:**  \n    An empty input bo

## Step 6: Process All Elements (Small Batch)

In [7]:
# Process a small batch to demonstrate
if summarizer and elements:
    # Take first 10 elements only
    sample_elements = elements[:10]
    
    print(f"Processing {len(sample_elements)} elements...\n")
    
    # Process with summarizer
    enriched_elements = summarizer.process_elements(sample_elements)
    
    print(f"\n✓ Processed {len(enriched_elements)} elements")
    
    # Show statistics
    stats = summarizer.get_statistics()
    
    print("\nProcessing Statistics:")
    print(f"  Text elements summarized: {stats['text_summarized']}")
    print(f"  Images described: {stats['images_described']}")
    print(f"  Code files processed: {stats['code_processed']}")
    print(f"  Tables summarized: {stats['tables_summarized']}")
    print(f"  Total API calls: {stats['total_api_calls']}")
    
    # Show sample enriched element
    if enriched_elements:
        sample = enriched_elements[0]
        print("\nSample enriched element:")
        print(f"  Type: {sample['type']}")
        if 'summary_short' in sample:
            print(f"  Short summary: {sample['summary_short']}")
        if 'summary_medium' in sample:
            print(f"  Medium summary: {sample['summary_medium'][:100]}...")

Processing 10 elements...


✓ Processed 10 elements

Processing Statistics:
  Text elements summarized: 7
  Images described: 1
  Code files processed: 0
  Tables summarized: 0
  Total API calls: 16

Sample enriched element:
  Type: text
  Short summary: JavaScript module loader setup and utility functions.
  Medium summary: This code includes references to several MadCap Software JavaScript files and embeds RequireJS v2.1....


## Step 7: Cost Estimation

In [8]:
# Estimate costs for full dataset
if summarizer and elements:
    print("Cost Estimation:\n")
    print("=" * 80)
    
    # Count element types
    type_counts = {}
    for elem in elements:
        elem_type = elem['type']
        type_counts[elem_type] = type_counts.get(elem_type, 0) + 1
    
    # GPT-4o-mini pricing (approximate)
    # Input: $0.15 / 1M tokens
    # Output: $0.60 / 1M tokens
    
    text_count = type_counts.get('text', 0)
    image_count = type_counts.get('image', 0)
    code_count = type_counts.get('code', 0)
    table_count = type_counts.get('table', 0)
    
    # Rough estimates
    # Text: ~500 input tokens, ~150 output tokens per element
    # Image: ~100 tokens (vision API cost)
    # Code: ~300 input, ~100 output
    # Table: ~200 input, ~80 output
    
    text_input_tokens = text_count * 500
    text_output_tokens = text_count * 150
    
    image_tokens = image_count * 100
    
    code_input_tokens = code_count * 300
    code_output_tokens = code_count * 100
    
    table_input_tokens = table_count * 200
    table_output_tokens = table_count * 80
    
    total_input = text_input_tokens + code_input_tokens + table_input_tokens
    total_output = text_output_tokens + code_output_tokens + table_output_tokens + image_tokens
    
    input_cost = (total_input / 1_000_000) * 0.15
    output_cost = (total_output / 1_000_000) * 0.60
    total_cost = input_cost + output_cost
    
    print(f"Elements to process:")
    print(f"  Text: {text_count}")
    print(f"  Images: {image_count}")
    print(f"  Code: {code_count}")
    print(f"  Tables: {table_count}")
    print(f"  Total: {len(elements)}")
    
    print(f"\nEstimated tokens:")
    print(f"  Input: ~{total_input:,}")
    print(f"  Output: ~{total_output:,}")
    
    print(f"\nEstimated cost (GPT-4o-mini):")
    print(f"  Input: ${input_cost:.4f}")
    print(f"  Output: ${output_cost:.4f}")
    print(f"  Total: ${total_cost:.4f}")
    
    print("\n" + "=" * 80)
    print("\n✓ Summarization demonstration complete!")
    print("\nNext: 04_qdrant_storage.ipynb - Vector storage and search")

Cost Estimation:

Elements to process:
  Text: 109066
  Images: 831
  Code: 0
  Tables: 13027
  Total: 123310

Estimated tokens:
  Input: ~57,138,400
  Output: ~17,485,160

Estimated cost (GPT-4o-mini):
  Input: $8.5708
  Output: $10.4911
  Total: $19.0619


✓ Summarization demonstration complete!

Next: 04_qdrant_storage.ipynb - Vector storage and search
